# 02 — Estatística descritiva

Produz painéis exatos anuais, mensais e por dimensões substantivas.

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import os
import subprocess
import sys
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/falando_nela/data")
REPO_DIR = Path("/content/falando_nela")
REPO_URL = "https://github.com/pedblan/falando_nela.git"
REPO_REF = ""  # Opcional: branch, tag ou commit; vazio acompanha o default remoto.

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags", "--prune"], check=True)
    if not REPO_REF:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
if REPO_REF:
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)

os.chdir(REPO_DIR)
os.environ["FALANDO_NELA_DATA_ROOT"] = str(DATA_ROOT)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "--force-reinstall",
        "--no-cache-dir",
        "numpy==2.0.2",
        "pandas==2.2.3",
    ],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-analise.txt"], check=True)
ABI_CHECK = subprocess.run(
    [
        sys.executable,
        "-c",
        (
            "import numpy as np; import pandas as pd; "
            "assert np.__version__ == '2.0.2', np.__version__; "
            "assert pd.__version__ == '2.2.3', pd.__version__; "
            "print(f'NumPy {np.__version__}; pandas {pd.__version__}')"
        ),
    ],
    check=True,
    text=True,
    capture_output=True,
)
import numpy as np
import pandas as pd

assert np.__version__ == "2.0.2", f"Reinicie a sessao do Colab: NumPy carregado={np.__version__}"
assert pd.__version__ == "2.2.3", f"Reinicie a sessao do Colab: pandas carregado={pd.__version__}"
print("Data root:", DATA_ROOT)
print("Commit:", subprocess.run(["git", "rev-parse", "HEAD"], check=True, text=True, capture_output=True).stdout.strip())
print("ABI:", ABI_CHECK.stdout.strip())

Data root: /content/drive/MyDrive/falando_nela/data
Commit: 42928c65c8d1d9b8ee43a2d67272236b4cb7ed79
ABI: NumPy 2.0.2; pandas 2.2.3


## Configuração

Use o mesmo `RUN_ID` em toda a suíte. A configuração versionada é a fonte de verdade.

In [3]:
from analise.discursos_plenario.config import load_config, resolve_input_paths, resolve_output_root

RUN_ID = "analise-plenario-20260717-v1"
CONFIG_PATH = REPO_DIR / "analise" / "discursos_plenario" / "config.v1.json"
ANALYSIS_CONFIG = load_config(CONFIG_PATH)
RUN_OUTPUT_ROOT = resolve_output_root(ANALYSIS_CONFIG, DATA_ROOT, RUN_ID)
INPUT_PATHS = resolve_input_paths(ANALYSIS_CONFIG, DATA_ROOT)
RODAR_ETAPA = True

assert ANALYSIS_CONFIG.date_start == "2010-02-02"
assert ANALYSIS_CONFIG.date_end == "2026-07-13"
assert ANALYSIS_CONFIG.raw["complete_year_end"] == 2025
assert ANALYSIS_CONFIG.raw["ytd_year"] == 2026
print("Run:", RUN_ID)
print("Saida:", RUN_OUTPUT_ROOT)

Run: analise-plenario-20260717-v1
Saida: /content/drive/MyDrive/falando_nela/data/analises/discursos_plenario/v1/analise-plenario-20260717-v1


## Decisão metodológica

Bootstrap não é automático: use-o apenas para uma pergunta que declare explicitamente sua população de generalização.

In [4]:
DESCRITIVAS_SNAPSHOT_PATH = RUN_OUTPUT_ROOT / "00_snapshot" / "discursos_plenario_snapshot.parquet"
assert DESCRITIVAS_SNAPSHOT_PATH.exists(), "Execute o caderno 00."

## Execução

A etapa cara permanece desativada até a inspeção das entradas e dos parâmetros acima.

In [5]:
from analise.discursos_plenario.descritivas import run_descriptives

DESCRITIVAS_RESULT = None
if RODAR_ETAPA:
    DESCRITIVAS_RESULT = run_descriptives(data_root=DATA_ROOT, run_id=RUN_ID, config_path=CONFIG_PATH)
    print(DESCRITIVAS_RESULT["manifest_path"])
else:
    print("Descritivas não executadas.")

/content/drive/MyDrive/falando_nela/data/analises/discursos_plenario/v1/analise-plenario-20260717-v1/02_descritivas/manifest.json


## Validação imediata

Esta checagem não substitui os testes sintéticos nem a revisão dos manifests.

In [7]:
import pandas as pd

DESCRITIVAS_ANUAL_PATH = RUN_OUTPUT_ROOT / "02_descritivas" / "anual.csv"
if DESCRITIVAS_ANUAL_PATH.exists():
    DESCRITIVAS_ANUAL = pd.read_csv(DESCRITIVAS_ANUAL_PATH)
    assert DESCRITIVAS_ANUAL["discursos"].ge(0).all()
    display(DESCRITIVAS_ANUAL.tail(50))

,arena,periodo,discursos,oradores,palavras,palavras_media,palavras_mediana,palavras_desvio_padrao,palavras_p25,palavras_p75,palavras_p90,discursos_por_orador,discursos_por_mil,diferenca_periodo_anterior,diferenca_mediana_historica
1,camara,2011,15908,538,8133051,511.255406,343.0,712.345557,158.00,618.00,1020.0,29.568773,1000.0,-155.0,-1756.0
2,camara,2012,20291,543,10337436,509.459169,349.0,678.730095,156.00,619.50,1018.0,37.368324,1000.0,4383.0,2627.0
3,camara,2013,28174,542,13270773,471.029069,308.0,646.241360,146.00,567.00,962.0,51.981550,1000.0,7883.0,10510.0
4,camara,2014,17664,529,8526848,482.724638,312.0,770.152615,146.00,570.25,993.0,33.391304,1000.0,-10510.0,0.0
5,camara,2015,28514,563,11401434,399.853896,261.0,608.673462,134.00,497.00,792.0,50.646536,1000.0,10850.0,10850.0
6,camara,2016,23633,593,9633028,407.609191,287.0,603.448127,141.00,506.00,811.0,39.853288,1000.0,-4881.0,5969.0
7,camara,2017,28222,586,10088208,357.459004,219.0,467.954737,128.00,449.75,737.0,48.160410,1000.0,4589.0,10558.0
8,camara,2018,15707,529,5594405,356.172726,208.0,406.271327,132.00,454.00,766.4,29.691871,1000.0,-12515.0,-1957.0
9,camara,2019,20322,539,7060295,347.421268,188.0,727.540292,130.00,427.00,707.0,37.703154,1000.0,4615.0,2658.0
10,camara,2020,14903,464,4545621,305.013823,191.0,314.520469,130.00,395.00,614.0,32.118534,1000.0,-5419.0,-2761.0
